# 🎯 Hyperparameter Optimization (HPO) for Time-Series Transformer
Automated Hyperparameter Tuning using **Optuna (Bayesian Optimization)** on Validation Set.
Finds optimal `d_model`, `num_heads`, `d_ff`, `num_layers`, `dropout_rate`, `l2_reg`, `learning_rate`, and `batch_size`.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Install optuna if not installed
try:
    import optuna
except ImportError:
    !pip install optuna
    import optuna

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("TensorFlow Version:", tf.__version__)
print("Optuna Version:", optuna.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow Version: 2.21.0
Optuna Version: 4.9.0
GPU Available: False


In [2]:
# Auto-detect dataset path
data_path = r'C:\Users\chaya\Documents\Program\Practice\preprocess\acn_caltech_ready2.csv'

df = pd.read_csv(data_path)
df['connectionTime'] = pd.to_datetime(df['connectionTime'])
df = df.set_index('connectionTime')

# Drop unneeded columns
df = df.drop(columns=['prcp', 'tempDiff_48', 'cldc'], errors='ignore')

cols = []
for col in df.columns:
    df[col] = df[col].astype('float32')
    if col != 'kWhDelivered':
        cols.append(col)

X = df[cols]
y = df['kWhDelivered']

print(f"Dataset Loaded successfully from {data_path}! Total Rows: {len(df)}, Features Count: {len(cols)}")

Dataset Loaded successfully from C:\Users\chaya\Documents\Program\Practice\preprocess\acn_caltech_ready2.csv! Total Rows: 32435, Features Count: 27


In [3]:
# Train/Val/Test Split (60% / 20% / 20%)
train_len = int(len(df) * 0.6)
val_len = int(len(df) * 0.2)

X_train = X[:train_len]
X_val   = X[train_len : train_len + val_len]
X_test  = X[train_len + val_len :]

y_train = y[:train_len]
y_val   = y[train_len : train_len + val_len]
y_test  = y[train_len + val_len :]

# Feature Scaling (MinMaxScaler)
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled   = scaler_X.transform(X_val)
X_test_scaled  = scaler_X.transform(X_test)

# Target Scaling (MinMaxScaler for y)
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled   = scaler_y.transform(y_val.values.reshape(-1, 1)).flatten()
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()

LOOKBACK = 96
HORIZON = 48
print(f"Data Prep Completed! LOOKBACK={LOOKBACK}, HORIZON={HORIZON}")

Data Prep Completed! LOOKBACK=96, HORIZON=48


In [4]:
# Helper 1: Windowed Dataset Creator
def create_windowed_dataset(X_data, y_data, lookback, horizon, batch_size=64, shuffle=True):
    X_seq, y_seq = [], []
    for i in range(len(X_data) - lookback - horizon + 1):
        X_seq.append(X_data[i : i + lookback])
        y_seq.append(y_data[i + lookback : i + lookback + horizon])
    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.float32)

    dataset = tf.data.Dataset.from_tensor_slices((X_seq, y_seq))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(X_seq))
    dataset = dataset.batch(batch_size, drop_remainder=shuffle).prefetch(tf.data.AUTOTUNE)
    return dataset, X_seq, y_seq

# Helper 2: Positional Embedding Layer
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, seq_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = tf.keras.layers.Embedding(input_dim=seq_len, output_dim=d_model)
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[1], delta=1)
        return x + self.pos_emb(positions)

# Helper 3: Baseline Encoder-Only Transformer Architecture
def build_encoder_only_transformer(lookback, num_features, horizon, d_model=64, num_heads=4, d_ff=128, num_layers=2,
                                   dropout_rate=0.2, l2_reg=1e-3, noise_stddev=0.05):
    inputs = tf.keras.layers.Input(shape=(lookback, num_features))
    x = tf.keras.layers.Dense(d_model, 
                                kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
                                bias_regularizer=tf.keras.regularizers.l2(l2_reg))(inputs)
    x = tf.keras.layers.GaussianNoise(noise_stddev)(x)
    x = PositionalEmbedding(seq_len=lookback, d_model=d_model)(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)

    for _ in range(num_layers):
        attention_output = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model, dropout=dropout_rate
        )(query=x, value=x, use_causal_mask=False)
        attention_output = tf.keras.layers.Dropout(dropout_rate)(attention_output)
        x = tf.keras.layers.Add()([x, attention_output])
        x = tf.keras.layers.LayerNormalization()(x)

        ffn_output = tf.keras.layers.Dense(d_ff, activation="relu",
                                           kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
                                           bias_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
        ffn_output = tf.keras.layers.Dense(d_model,
                                           kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
                                           bias_regularizer=tf.keras.regularizers.l2(l2_reg))(ffn_output)
        ffn_output = tf.keras.layers.Dropout(dropout_rate)(ffn_output)
        x = tf.keras.layers.Add()([x, ffn_output])
        x = tf.keras.layers.LayerNormalization()(x)

    last_step_feat = x[:, -1, :]
    global_avg_feat = tf.keras.layers.GlobalAveragePooling1D()(x)
    history_context = tf.keras.layers.Concatenate()([last_step_feat, global_avg_feat])

    x = tf.keras.layers.Dense(128, activation="relu",
                              kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
                              bias_regularizer=tf.keras.regularizers.l2(l2_reg))(history_context)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Dense(64, activation="relu",
                              kernel_regularizer=tf.keras.regularizers.l2(l2_reg),
                              bias_regularizer=tf.keras.regularizers.l2(l2_reg))(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(horizon)(x)
    model = tf.keras.Model(inputs, outputs, name="EncoderOnlyTimeSeriesTransformer")
    return model

In [5]:
# Optuna Objective Function
def objective(trial):
    # 1. Define Search Space for Hyperparameters
    d_model       = trial.suggest_categorical('d_model', [32, 64, 128])
    num_heads     = trial.suggest_categorical('num_heads', [2, 4, 8])
    d_ff          = trial.suggest_categorical('d_ff', [64, 128, 256])
    num_layers    = trial.suggest_int('num_layers', 1, 3)
    dropout_rate  = trial.suggest_float('dropout_rate', 0.05, 0.2, step=0.05)
    l2_reg        = trial.suggest_float('l2_reg', 1e-4, 1e-2, log=True)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [64, 128, 256])
    
    # Build Windowed Datasets for this trial's batch size
    train_ds, _, _ = create_windowed_dataset(X_train_scaled, y_train_scaled, LOOKBACK, HORIZON, batch_size=batch_size, shuffle=True)
    val_ds, _, _   = create_windowed_dataset(X_val_scaled, y_val_scaled, LOOKBACK, HORIZON, batch_size=batch_size, shuffle=False)
    
    # Build Model with Trial Parameters
    model = build_encoder_only_transformer(
        lookback=LOOKBACK, 
        num_features=X_train_scaled.shape[1], 
        horizon=HORIZON,
        d_model=d_model, 
        num_heads=num_heads, 
        d_ff=d_ff, 
        num_layers=num_layers,
        dropout_rate=dropout_rate, 
        l2_reg=l2_reg
    )
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mse')
    
    # Early Stopping Callback
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    # Train for 20 epochs per trial
    history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop], verbose=1)
    
    best_val_loss = min(history.history['val_loss'])
    return best_val_loss

# Start Optuna Study (20 Trials)
print("Starting Optuna Hyperparameter Optimization Study (20 Trials)...")
study = optuna.create_study(direction="minimize", study_name="transformer_hpo")
study.optimize(objective, n_trials=20)

print("\n" + "="*60)
print("BEST HYPERPARAMETERS FOUND BY OPTUNA:")
print("="*60)
for key, val in study.best_params.items():
    print(f"  - {key:<15}: {val}")
print(f"\n  - Lowest Val Loss: {study.best_value:.6f}")
print("="*60)

[I 2026-08-10 23:54:47,234] A new study created in memory with name: transformer_hpo


Starting Optuna Hyperparameter Optimization Study (20 Trials)...

Epoch 1/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 24s 132ms/step - loss: 0.1779 - val_loss: 0.1386
Epoch 2/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 20s 130ms/step - loss: 0.1373 - val_loss: 0.1173
Epoch 3/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 20s 136ms/step - loss: 0.1177 - val_loss: 0.0998
Epoch 4/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 20s 133ms/step - loss: 0.1008 - val_loss: 0.0860
Epoch 5/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 20s 133ms/step - loss: 0.0869 - val_loss: 0.0748
Epoch 6/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 19s 127ms/step - loss: 0.0762 - val_loss: 0.0657
Epoch 7/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 20s 131ms/step - loss: 0.0673 - val_loss: 0.0579
Epoch 8/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 19s 125ms/step - loss: 0.0597 - val_loss: 0.0515
Epoch 9/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 18s 123ms/step - loss: 0.0532 - val_loss: 0.0456
Epoch 10/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 19s 128ms/step - loss: 0.0475 - val_loss: 0.0406
Epoch 11/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 19

[I 2026-08-11 00:01:19,267] Trial 0 finished with value: 0.013628014363348484 and parameters: {'d_model': 64, 'num_heads': 2, 'd_ff': 128, 'num_layers': 2, 'dropout_rate': 0.05, 'l2_reg': 0.0002363629767313072, 'learning_rate': 0.00021785345160242516, 'batch_size': 128}. Best is trial 0 with value: 0.013628014363348484.


Epoch 1/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 197s 631ms/step - loss: 0.1618 - val_loss: 0.1204
Epoch 2/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 187s 622ms/step - loss: 0.1179 - val_loss: 0.0942
Epoch 3/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 186s 619ms/step - loss: 0.0938 - val_loss: 0.0736
Epoch 4/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 194s 644ms/step - loss: 0.0741 - val_loss: 0.0554
Epoch 5/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 194s 646ms/step - loss: 0.0554 - val_loss: 0.0415
Epoch 6/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 204s 677ms/step - loss: 0.0426 - val_loss: 0.0316
Epoch 7/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 192s 639ms/step - loss: 0.0333 - val_loss: 0.0243
Epoch 8/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 193s 641ms/step - loss: 0.0265 - val_loss: 0.0189
Epoch 9/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 193s 640ms/step - loss: 0.0213 - val_loss: 0.0154
Epoch 10/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 193s 641ms/step - loss: 0.0174 - val_loss: 0.0123
Epoch 11/20
301/301 ━━━━━━━━━━━━━━━━━━━━ 193s 640ms/step - loss: 0.0147 - val_loss: 0.0104
Epoch 12

[I 2026-08-11 01:06:54,322] Trial 1 finished with value: 0.004778300411999226 and parameters: {'d_model': 128, 'num_heads': 8, 'd_ff': 64, 'num_layers': 3, 'dropout_rate': 0.2, 'l2_reg': 0.00016907542617390886, 'learning_rate': 0.00032117660717819585, 'batch_size': 64}. Best is trial 1 with value: 0.004778300411999226.


...

Epoch 1/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 348s 2s/step - loss: 0.1638 - val_loss: 0.1287
Epoch 2/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 333s 2s/step - loss: 0.1247 - val_loss: 0.1044
Epoch 3/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 319s 2s/step - loss: 0.1070 - val_loss: 0.0895
Epoch 4/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 316s 2s/step - loss: 0.0916 - val_loss: 0.0747
Epoch 5/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 264s 2s/step - loss: 0.0756 - val_loss: 0.0631
Epoch 6/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 328s 2s/step - loss: 0.0646 - val_loss: 0.0543
Epoch 7/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 346s 2s/step - loss: 0.0555 - val_loss: 0.0468
Epoch 8/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 341s 2s/step - loss: 0.0481 - val_loss: 0.0408
Epoch 9/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 340s 2s/step - loss: 0.0421 - val_loss: 0.0356
Epoch 10/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 344s 2s/step - loss: 0.0369 - val_loss: 0.0314
Epoch 11/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 334s 2s/step - loss: 0.0326 - val_loss: 0.0281
Epoch 12/20
150/150 ━━━━━━━━━━━━━━━━━━━━ 

[I 2026-08-11 12:23:47,445] Trial 18 finished with value: 0.01040618121623993 and parameters: {'d_model': 128, 'num_heads': 8, 'd_ff': 256, 'num_layers': 3, 'dropout_rate': 0.1, 'l2_reg': 0.00010205743728392336, 'learning_rate': 0.00045420947428129643, 'batch_size': 128}. Best is trial 15 with value: 0.0041544935666024685.


Epoch 1/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 238s 3s/step - loss: 0.5432 - val_loss: 0.3990
Epoch 2/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 229s 3s/step - loss: 0.3334 - val_loss: 0.2656
Epoch 3/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 197s 3s/step - loss: 0.2296 - val_loss: 0.1842
Epoch 4/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - loss: 0.1672 - val_loss: 0.1342
Epoch 5/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 211s 3s/step - loss: 0.1242 - val_loss: 0.0982
Epoch 6/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 218s 3s/step - loss: 0.0906 - val_loss: 0.0726
Epoch 7/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 217s 3s/step - loss: 0.0686 - val_loss: 0.0549
Epoch 8/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 220s 3s/step - loss: 0.0527 - val_loss: 0.0429
Epoch 9/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 204s 3s/step - loss: 0.0414 - val_loss: 0.0334
Epoch 10/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 164s 2s/step - loss: 0.0332 - val_loss: 0.0268
Epoch 11/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 161s 2s/step - loss: 0.0271 - val_loss: 0.0224
Epoch 12/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 173s 2s/step - loss: 0.0

[I 2026-08-11 13:28:17,680] Trial 19 finished with value: 0.008797723799943924 and parameters: {'d_model': 128, 'num_heads': 8, 'd_ff': 256, 'num_layers': 2, 'dropout_rate': 0.05, 'l2_reg': 0.0006293917869959169, 'learning_rate': 0.0009860919445185022, 'batch_size': 256}. Best is trial 15 with value: 0.0041544935666024685.



BEST HYPERPARAMETERS FOUND BY OPTUNA:
  - d_model        : 128
  - num_heads      : 8
  - d_ff           : 256
  - num_layers     : 3
  - dropout_rate   : 0.1
  - l2_reg         : 0.00021323727139843735
  - learning_rate  : 0.0009994764292692345
  - batch_size     : 64

  - Lowest Val Loss: 0.004154
